In [1]:
import os
os.chdir("/home/menta.sa/Debias_VLMs")
print("Working directory:", os.getcwd())

import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    print(f"GPU: {name}")
else:
    print("NO GPU — restart your session with a GPU!")

Working directory: /home/menta.sa/Debias_VLMs
GPU: NVIDIA A100-SXM4-40GB


In [2]:
import trl, transformers
print(f"trl: {trl.__version__}")
print(f"transformers: {transformers.__version__}")

trl: 0.29.1
transformers: 5.4.0


In [3]:
# !pip install -r requirements.txt

In [6]:
!python profile_pipeline.py

=== Pipeline Profiler ===
Total raw examples in dataset: 14578
Total preference pairs to process: 29156

--- Running Step 1: Embeddings (Smallset) ---
ERROR:__main__:Error in main execution: Some specified arguments are not used by the HfArgumentParser: ['cuda', 'Qwen/Qwen2-VL-2B-Instruct']
Traceback (most recent call last):
  File "/home/menta.sa/Debias_VLMs/profile_pipeline.py", line 238, in <module>
    main()
  File "/home/menta.sa/Debias_VLMs/profile_pipeline.py", line 118, in main
    subprocess.run(cmd1, check=True)
  File "/shared/EL9/explorer/anaconda3/2024.06/lib/python3.12/subprocess.py", line 571, in run
    raise CalledProcessError(retcode, process.args,
subprocess.CalledProcessError: Command '['python', 'cal_emb_modular.py', '--device', 'mps', 'cuda', 'Qwen/Qwen2-VL-2B-Instruct', '--data_path', './sb_bench_data/data', '--cls_embs_path', './embeddings_output', '--batch_size', '1', '--use_smallset']' returned non-zero exit status 1.
^C
Exception ignored in: <module 'threadi

In [7]:
import subprocess, sys, os, time, glob
import pandas as pd

python_cmd = sys.executable
data_path = "./sb_bench_data/data"
emb_dir = "./embeddings_output"
model_id = "Qwen/Qwen2-VL-2B-Instruct"

# Count dataset
total_raw = sum(len(pd.read_parquet(f, engine="fastparquet")) 
                for f in glob.glob(os.path.join(data_path, "*.parquet")))
total_pairs = total_raw * 2
print(f"Dataset: {total_raw} raw → {total_pairs} pairs")

# Clean old embeddings
for f in glob.glob(f"{emb_dir}/emb_*.npy"):
    os.remove(f)

# STEP 1
print("\n=== STEP 1: Embeddings ===")
t0 = time.time()
subprocess.run([
    python_cmd, "cal_emb_modular.py",
    "--device", "cuda",
    "--model", model_id,
    "--data_path", data_path,
    "--cls_embs_path", emb_dir,
    "--batch_size", "1",
    "--use_smallset"
], check=True)
step1 = time.time() - t0
print(f"Step 1: {step1:.1f}s | Est full: {(step1/10)*total_pairs/3600:.2f} hrs")

# STEP 2
print("\n=== STEP 2: PCA ===")
t0 = time.time()
subprocess.run([
    python_cmd, "generate_drm_heads.py",
    "--input_dir", emb_dir,
    "--output_dir", "./generated_heads",
    "--n_components", "5",
    "--case_name", "sb_bench"
], check=True)
step2 = time.time() - t0
print(f"Step 2: {step2:.1f}s | Est full: {step2*(total_pairs/10)/3600:.2f} hrs")

# STEP 3
print("\n=== STEP 3: Evaluate ===")
t0 = time.time()
subprocess.run([
    python_cmd, "evaluate_drm_heads.py",
    "--emb_dir", emb_dir,
    "--score_head_weight", "./generated_heads/sb_bench-PCA-component",
    "--data_path", data_path,
    "--output_json", "./drm_head_results.json",
    "--device", "cuda"
], check=True)
step3 = time.time() - t0
print(f"Step 3: {step3:.1f}s | Est full: {step3*(total_pairs/10)/3600:.2f} hrs")

total_hrs = ((step1/10)*total_pairs + step2*(total_pairs/10) + step3*(total_pairs/10)) / 3600
print(f"\n{'='*50}")
print(f"TOTAL ESTIMATED: {total_hrs:.2f} hours")

Dataset: 14578 raw → 29156 pairs

=== STEP 1: Embeddings ===


INFO:modules.model_loader:Using device: cuda
INFO:modules.model_loader:Using dtype: torch.bfloat16
INFO:modules.model_loader:Using attention implementation: eager
INFO:modules.model_loader:✅ Local model configuration loaded
INFO:__main__:Loading model and processor...
INFO:modules.model_loader:Attempting to load model: Qwen/Qwen2-VL-2B-Instruct
INFO:modules.model_loader:Using Qwen2VL model class


Model Qwen/Qwen2-VL-2B-Instruct not found locally, using HuggingFace
Model Qwen/Qwen2-VL-2B-Instruct not found locally, using HuggingFace


INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-VL-2B-Instruct/895c3a49bc3fa70a340399125c650a463535e71c/preprocessor_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-VL-2B-Instruct/895c3a49bc3fa70a340399125c650a463535e71c/preprocessor_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://hugging

Model Qwen/Qwen2-VL-2B-Instruct not found locally, using HuggingFace


INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-VL-2B-Instruct/895c3a49bc3fa70a340399125c650a463535e71c/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-VL-2B-Instruct/895c3a49bc3fa70a340399125c650a463535e71c/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct/resolve/main/model.safetensors.index.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Q

CalledProcessError: Command '['/shared/EL9/explorer/anaconda3/2024.06/bin/python', 'cal_emb_modular.py', '--device', 'cuda', '--model', 'Qwen/Qwen2-VL-2B-Instruct', '--data_path', './sb_bench_data/data', '--cls_embs_path', './embeddings_output', '--batch_size', '1', '--use_smallset']' returned non-zero exit status 1.